# RetailPulse -- Customer Segmentation

**Objective:** Segment customers using K-Means and DBSCAN on RFM features. Evaluate clustering quality with Silhouette Score, Calinski-Harabasz Index, and Davies-Bouldin Index. Assign business-meaningful labels to each cluster.


In [1]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import (calinski_harabasz_score, davies_bouldin_score,
                             silhouette_samples, silhouette_score)
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

FIGURES_DIR = os.path.join("..", "reports", "figures")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(FIGURES_DIR, exist_ok=True)


In [2]:
def save_fig(fig, name):
    path = os.path.join(FIGURES_DIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Saved: {name}")


## Load RFM Data

In [3]:
rfm = pd.read_csv(os.path.join(PROCESSED_DIR, "customer_rfm.csv"))
print(f"Customers: {len(rfm):,}")
rfm.head()


Customers: 4,312


,Customer ID,recency,frequency,monetary,r_score,f_score,m_score,rfm_score,rfm_segment,segment_label
0,12346,165,11,372.86,2,5,2,9,252,Others
1,12347,3,2,1323.32,5,2,4,11,524,New Customers
2,12348,74,1,222.16,2,1,1,4,211,Lost
3,12349,43,3,2671.14,3,3,5,11,335,Loyal Customers
4,12351,11,1,300.93,5,1,2,8,512,New Customers


## Feature Preparation

Log-transform the skewed RFM features, then standardize to zero mean and unit variance. This ensures clustering algorithms are not dominated by large-scale variables.


In [4]:
# Log-transform to reduce skewness (add 1 to handle zeros)
rfm_features = rfm[["recency", "frequency", "monetary"]].copy()
rfm_log = np.log1p(rfm_features)

# Standardize
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)
rfm_scaled = pd.DataFrame(rfm_scaled, columns=["recency", "frequency", "monetary"])

print("Before scaling (log-transformed):")
print(rfm_log.describe().round(3).to_string())
print()
print("After scaling:")
print(rfm_scaled.describe().round(3).to_string())


Before scaling (log-transformed):
        recency  frequency  monetary
count  4312.000   4312.000  4312.000
mean      3.856      1.371     6.609
std       1.300      0.691     1.281
min       0.693      0.693     1.374
25%       2.944      0.693     5.731
50%       3.989      1.099     6.555
75%       4.920      1.792     7.448
max       5.927      5.328    12.763

After scaling:
        recency  frequency  monetary
count  4312.000   4312.000  4312.000
mean     -0.000      0.000     0.000
std       1.000      1.000     1.000
min      -2.433     -0.982    -4.087
25%      -0.701     -0.982    -0.686
50%       0.102     -0.395    -0.042
75%       0.819      0.609     0.655
max       1.593      5.730     4.804


## Finding Optimal Number of Clusters

Using the Elbow method (inertia) and Silhouette Score to determine the best K for K-Means.


In [5]:
K_range = range(2, 11)
inertias = []
silhouette_scores = []
calinski_scores = []
davies_scores = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    labels = km.fit_predict(rfm_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(rfm_scaled, labels))
    calinski_scores.append(calinski_harabasz_score(rfm_scaled, labels))
    davies_scores.append(davies_bouldin_score(rfm_scaled, labels))

print(f"{'K':>3}  {'Inertia':>10}  {'Silhouette':>10}  {'Calinski-H':>12}  {'Davies-B':>10}")
print("-" * 55)
for i, k in enumerate(K_range):
    print(f"{k:>3}  {inertias[i]:>10.1f}  {silhouette_scores[i]:>10.4f}  "
          f"{calinski_scores[i]:>12.1f}  {davies_scores[i]:>10.4f}")


  K     Inertia  Silhouette    Calinski-H    Davies-B
-------------------------------------------------------
  2      6605.3      0.4223        4130.8      0.9108
  3      4973.8      0.3284        3449.0      1.0662
  4      3977.4      0.3314        3234.4      1.0115
  5      3347.9      0.3192        3083.7      1.0055
  6      2925.0      0.3119        2947.5      1.0018
  7      2612.1      0.3075        2835.8      0.9599
  8      2397.0      0.2789        2703.4      0.9922
  9      2212.6      0.2762        2606.9      1.0208
 10      2042.6      0.2795        2549.3      0.9987


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Elbow curve
axes[0].plot(list(K_range), inertias, "bo-", linewidth=2, markersize=8)
axes[0].set_xlabel("Number of Clusters (K)")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow Method")
axes[0].set_xticks(list(K_range))

# Silhouette score
axes[1].plot(list(K_range), silhouette_scores, "rs-", linewidth=2, markersize=8)
axes[1].set_xlabel("Number of Clusters (K)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Analysis")
axes[1].set_xticks(list(K_range))
best_k_sil = list(K_range)[np.argmax(silhouette_scores)]
axes[1].axvline(best_k_sil, color="#2ecc71", linestyle="--", alpha=0.7,
                label=f"Best K={best_k_sil}")
axes[1].legend()

# Calinski-Harabasz
axes[2].plot(list(K_range), calinski_scores, "g^-", linewidth=2, markersize=8)
axes[2].set_xlabel("Number of Clusters (K)")
axes[2].set_ylabel("Calinski-Harabasz Index")
axes[2].set_title("Calinski-Harabasz Score")
axes[2].set_xticks(list(K_range))

fig.suptitle("Cluster Evaluation Metrics", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "11_cluster_evaluation_metrics.png")
plt.show()


Saved: 11_cluster_evaluation_metrics.png


## K-Means Clustering

Based on the elbow curve and silhouette analysis, we select the optimal K and fit the final K-Means model.


In [7]:
# Select optimal K based on silhouette score
optimal_k = list(K_range)[np.argmax(silhouette_scores)]
print(f"Optimal K (by silhouette): {optimal_k}")
print(f"Silhouette score: {max(silhouette_scores):.4f}")

kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10, max_iter=300)
rfm["kmeans_cluster"] = kmeans.fit_predict(rfm_scaled)

# Cluster summary
cluster_summary = rfm.groupby("kmeans_cluster").agg(
    count=("Customer ID", "count"),
    avg_recency=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean"),
    median_recency=("recency", "median"),
    median_frequency=("frequency", "median"),
    median_monetary=("monetary", "median"),
).round(2)

cluster_summary["pct"] = (cluster_summary["count"] / len(rfm) * 100).round(1)
print()
print("K-Means Cluster Summary:")
print(cluster_summary.to_string())


Optimal K (by silhouette): 2
Silhouette score: 0.4223

K-Means Cluster Summary:
                count  avg_recency  avg_frequency  avg_monetary  median_recency  median_frequency  median_monetary   pct
kmeans_cluster                                                                                                          
0                2670       128.99           1.73        517.66            91.0               1.0           369.42  61.9
1                1642        29.68           8.88       4516.50            17.0               6.0          2154.18  38.1


In [8]:
# Rank clusters by monetary value to assign meaningful labels
cluster_monetary_rank = (cluster_summary["avg_monetary"]
                         .sort_values(ascending=False)
                         .index.tolist())

# Define label templates based on spending + recency patterns
def assign_kmeans_labels(rfm_df, cluster_col):
    cluster_profiles = rfm_df.groupby(cluster_col).agg(
        avg_r=("recency", "mean"),
        avg_f=("frequency", "mean"),
        avg_m=("monetary", "mean"),
    )

    labels = {}
    for cluster_id in cluster_profiles.index:
        r = cluster_profiles.loc[cluster_id, "avg_r"]
        f = cluster_profiles.loc[cluster_id, "avg_f"]
        m = cluster_profiles.loc[cluster_id, "avg_m"]

        med_r = rfm_df["recency"].median()
        med_f = rfm_df["frequency"].median()
        med_m = rfm_df["monetary"].median()

        if f > med_f * 2 and m > med_m * 2:
            labels[cluster_id] = "VIP / Champions"
        elif r < med_r and f > med_f:
            labels[cluster_id] = "Loyal Regulars"
        elif r < med_r and f <= med_f:
            labels[cluster_id] = "Recent Buyers"
        elif r > med_r * 1.5 and m > med_m:
            labels[cluster_id] = "Lapsing High-Value"
        elif r > med_r and f <= med_f:
            labels[cluster_id] = "Dormant"
        else:
            labels[cluster_id] = "Mid-Tier"

    return labels

kmeans_labels = assign_kmeans_labels(rfm, "kmeans_cluster")
rfm["kmeans_label"] = rfm["kmeans_cluster"].map(kmeans_labels)

print("K-Means Cluster Labels:")
print("-" * 45)
for cid, label in sorted(kmeans_labels.items()):
    count = (rfm["kmeans_cluster"] == cid).sum()
    print(f"  Cluster {cid}: {label:<25s} ({count:,} customers)")


K-Means Cluster Labels:
---------------------------------------------
  Cluster 0: Dormant                   (2,670 customers)
  Cluster 1: VIP / Champions           (1,642 customers)


In [9]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

colors = plt.cm.Set2(np.linspace(0, 1, optimal_k))
color_map = {c: colors[i] for i, c in enumerate(sorted(rfm["kmeans_cluster"].unique()))}

# Recency vs Frequency
for cluster in sorted(rfm["kmeans_cluster"].unique()):
    mask = rfm["kmeans_cluster"] == cluster
    axes[0].scatter(rfm.loc[mask, "recency"], rfm.loc[mask, "frequency"],
                    c=[color_map[cluster]], label=kmeans_labels[cluster],
                    alpha=0.5, s=20, edgecolors="none")
axes[0].set_xlabel("Recency (days)")
axes[0].set_ylabel("Frequency (orders)")
axes[0].set_title("Recency vs Frequency")
axes[0].legend(fontsize=8, loc="upper right")

# Frequency vs Monetary
for cluster in sorted(rfm["kmeans_cluster"].unique()):
    mask = rfm["kmeans_cluster"] == cluster
    axes[1].scatter(rfm.loc[mask, "frequency"],
                    rfm.loc[mask, "monetary"].clip(upper=rfm["monetary"].quantile(0.98)),
                    c=[color_map[cluster]], label=kmeans_labels[cluster],
                    alpha=0.5, s=20, edgecolors="none")
axes[1].set_xlabel("Frequency (orders)")
axes[1].set_ylabel("Monetary (revenue)")
axes[1].set_title("Frequency vs Monetary")

# Recency vs Monetary
for cluster in sorted(rfm["kmeans_cluster"].unique()):
    mask = rfm["kmeans_cluster"] == cluster
    axes[2].scatter(rfm.loc[mask, "recency"],
                    rfm.loc[mask, "monetary"].clip(upper=rfm["monetary"].quantile(0.98)),
                    c=[color_map[cluster]], label=kmeans_labels[cluster],
                    alpha=0.5, s=20, edgecolors="none")
axes[2].set_xlabel("Recency (days)")
axes[2].set_ylabel("Monetary (revenue)")
axes[2].set_title("Recency vs Monetary")

fig.suptitle("K-Means Clustering -- RFM Feature Space", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "12_kmeans_clusters.png")
plt.show()


Saved: 12_kmeans_clusters.png


In [10]:
# Normalized cluster profile heatmap
profile = rfm.groupby("kmeans_cluster")[["recency", "frequency", "monetary"]].mean()
profile_norm = (profile - profile.min()) / (profile.max() - profile.min())
profile_norm.index = [kmeans_labels[i] for i in profile_norm.index]

fig, ax = plt.subplots(figsize=(10, max(4, optimal_k * 0.8)))
sns.heatmap(profile_norm, annot=profile.values.round(1), fmt="g",
            cmap="YlOrRd", linewidths=1, ax=ax,
            xticklabels=["Recency (days)", "Frequency (orders)", "Monetary (revenue)"],
            cbar_kws={"label": "Normalized Value"})
ax.set_title("K-Means Cluster Profiles (annotated with actual values)",
             fontsize=14, fontweight="bold", pad=15)
ax.set_ylabel("")
fig.tight_layout()
save_fig(fig, "13_kmeans_cluster_profiles.png")
plt.show()


Saved: 13_kmeans_cluster_profiles.png


## DBSCAN Clustering

DBSCAN is a density-based algorithm that can find arbitrarily shaped clusters and automatically identifies outliers as noise points. We tune `eps` and `min_samples` parameters.


In [11]:
# Find suitable eps using k-distance graph
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(n_neighbors=5)
nn.fit(rfm_scaled)
distances, _ = nn.kneighbors(rfm_scaled)
k_distances = np.sort(distances[:, 4])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(len(k_distances)), k_distances, color="#3498db", linewidth=1)
ax.set_xlabel("Points (sorted by distance)")
ax.set_ylabel("5th Nearest Neighbor Distance")
ax.set_title("K-Distance Graph for DBSCAN eps Selection")

# Mark the elbow region
elbow_idx = int(len(k_distances) * 0.95)
ax.axhline(k_distances[elbow_idx], color="#e74c3c", linestyle="--", alpha=0.7,
           label=f"Suggested eps ~ {k_distances[elbow_idx]:.2f}")
ax.legend()
fig.tight_layout()
save_fig(fig, "14_dbscan_kdistance.png")
plt.show()

print(f"Suggested eps (95th percentile of 5-NN distances): {k_distances[elbow_idx]:.3f}")


Saved: 14_dbscan_kdistance.png
Suggested eps (95th percentile of 5-NN distances): 0.378


In [12]:
# Test multiple eps values
eps_candidates = np.arange(0.3, 1.5, 0.1)
dbscan_results = []

for eps in eps_candidates:
    for min_samp in [5, 10, 15]:
        db = DBSCAN(eps=eps, min_samples=min_samp)
        labels = db.fit_predict(rfm_scaled)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        noise_pct = (labels == -1).sum() / len(labels) * 100

        if n_clusters >= 2:
            sil = silhouette_score(rfm_scaled[labels != -1], labels[labels != -1])
        else:
            sil = -1

        dbscan_results.append({
            "eps": round(eps, 2),
            "min_samples": min_samp,
            "n_clusters": n_clusters,
            "noise_pct": round(noise_pct, 1),
            "silhouette": round(sil, 4)
        })

results_df = pd.DataFrame(dbscan_results)
# Show only configs with 3-8 clusters and reasonable noise
valid = results_df[(results_df["n_clusters"].between(3, 8)) & (results_df["noise_pct"] < 30)]
print("DBSCAN Parameter Search (filtered: 3-8 clusters, <30% noise):")
if len(valid) > 0:
    print(valid.sort_values("silhouette", ascending=False).head(10).to_string(index=False))
else:
    print("No valid configs found. Showing top results:")
    print(results_df[results_df["n_clusters"] >= 2]
          .sort_values("silhouette", ascending=False).head(10).to_string(index=False))


DBSCAN Parameter Search (filtered: 3-8 clusters, <30% noise):
 eps  min_samples  n_clusters  noise_pct  silhouette
 0.5           10           3        1.9      0.2551
 0.4           15           3        6.6      0.1701
 0.3           15           4       15.4      0.1451
 0.4           10           4        4.8      0.1237
 0.4            5           5        2.5      0.0868
 0.3           10           6       11.1      0.0775


In [13]:
# Select best DBSCAN config
if len(valid) > 0:
    best_row = valid.sort_values("silhouette", ascending=False).iloc[0]
else:
    best_row = results_df[results_df["n_clusters"] >= 2].sort_values(
        "silhouette", ascending=False).iloc[0]

best_eps = best_row["eps"]
best_min_samples = int(best_row["min_samples"])
print(f"Best DBSCAN: eps={best_eps}, min_samples={best_min_samples}")

dbscan = DBSCAN(eps=best_eps, min_samples=best_min_samples)
rfm["dbscan_cluster"] = dbscan.fit_predict(rfm_scaled)

n_clusters = rfm["dbscan_cluster"].nunique() - (1 if -1 in rfm["dbscan_cluster"].values else 0)
noise_count = (rfm["dbscan_cluster"] == -1).sum()
print(f"Clusters found: {n_clusters}")
print(f"Noise points: {noise_count} ({noise_count/len(rfm)*100:.1f}%)")
print()

# DBSCAN cluster summary
db_summary = rfm.groupby("dbscan_cluster").agg(
    count=("Customer ID", "count"),
    avg_recency=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean"),
).round(2)
db_summary["pct"] = (db_summary["count"] / len(rfm) * 100).round(1)
print("DBSCAN Cluster Summary:")
print(db_summary.to_string())


Best DBSCAN: eps=0.5, min_samples=10
Clusters found: 3
Noise points: 80 (1.9%)

DBSCAN Cluster Summary:
                count  avg_recency  avg_frequency  avg_monetary   pct
dbscan_cluster                                                       
-1                 80        81.61          29.58      30169.41   1.9
 0               2802        61.68           5.20       1927.33  65.0
 1               1409       151.65           1.00        326.18  32.7
 2                 21         5.14          41.05      24986.27   0.5


In [14]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

unique_clusters = sorted(rfm["dbscan_cluster"].unique())
cmap_colors = plt.cm.Set2(np.linspace(0, 1, max(len(unique_clusters), 3)))

for i, cluster in enumerate(unique_clusters):
    mask = rfm["dbscan_cluster"] == cluster
    label = f"Noise ({mask.sum():,})" if cluster == -1 else f"Cluster {cluster} ({mask.sum():,})"
    color = "#bdc3c7" if cluster == -1 else cmap_colors[i % len(cmap_colors)]
    alpha = 0.2 if cluster == -1 else 0.6
    size = 10 if cluster == -1 else 25

    axes[0].scatter(rfm.loc[mask, "recency"], rfm.loc[mask, "frequency"],
                    c=[color], label=label, alpha=alpha, s=size, edgecolors="none")
    axes[1].scatter(rfm.loc[mask, "frequency"],
                    rfm.loc[mask, "monetary"].clip(upper=rfm["monetary"].quantile(0.98)),
                    c=[color], label=label, alpha=alpha, s=size, edgecolors="none")

axes[0].set_xlabel("Recency (days)")
axes[0].set_ylabel("Frequency (orders)")
axes[0].set_title("Recency vs Frequency")
axes[0].legend(fontsize=8)

axes[1].set_xlabel("Frequency (orders)")
axes[1].set_ylabel("Monetary (revenue)")
axes[1].set_title("Frequency vs Monetary")
axes[1].legend(fontsize=8)

fig.suptitle("DBSCAN Clustering -- RFM Feature Space", fontsize=16, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "15_dbscan_clusters.png")
plt.show()


Saved: 15_dbscan_clusters.png


## Model Comparison

In [15]:
# Compare K-Means vs DBSCAN
comparison = {}

# K-Means metrics
km_labels = rfm["kmeans_cluster"].values
comparison["K-Means"] = {
    "n_clusters": optimal_k,
    "noise_points": 0,
    "silhouette": silhouette_score(rfm_scaled, km_labels),
    "calinski_harabasz": calinski_harabasz_score(rfm_scaled, km_labels),
    "davies_bouldin": davies_bouldin_score(rfm_scaled, km_labels),
}

# DBSCAN metrics (excluding noise)
db_labels = rfm["dbscan_cluster"].values
non_noise = db_labels != -1
if non_noise.sum() > 0 and len(set(db_labels[non_noise])) >= 2:
    comparison["DBSCAN"] = {
        "n_clusters": n_clusters,
        "noise_points": noise_count,
        "silhouette": silhouette_score(rfm_scaled[non_noise], db_labels[non_noise]),
        "calinski_harabasz": calinski_harabasz_score(rfm_scaled[non_noise], db_labels[non_noise]),
        "davies_bouldin": davies_bouldin_score(rfm_scaled[non_noise], db_labels[non_noise]),
    }
else:
    comparison["DBSCAN"] = {
        "n_clusters": n_clusters,
        "noise_points": noise_count,
        "silhouette": -1,
        "calinski_harabasz": -1,
        "davies_bouldin": -1,
    }

comp_df = pd.DataFrame(comparison).T
print("CLUSTERING MODEL COMPARISON")
print("=" * 65)
print(comp_df.round(4).to_string())
print()

# Interpretation
print("Interpretation:")
print("  Silhouette Score: higher is better (range -1 to 1)")
print("  Calinski-Harabasz: higher is better")
print("  Davies-Bouldin: lower is better")


CLUSTERING MODEL COMPARISON
         n_clusters  noise_points  silhouette  calinski_harabasz  davies_bouldin
K-Means         2.0           0.0      0.4223          4130.7979          0.9108
DBSCAN          3.0          80.0      0.2551          1325.9396          0.8575

Interpretation:
  Silhouette Score: higher is better (range -1 to 1)
  Calinski-Harabasz: higher is better
  Davies-Bouldin: lower is better


In [16]:
# Silhouette plot for K-Means
fig, ax = plt.subplots(figsize=(10, 7))

sample_silhouette = silhouette_samples(rfm_scaled, km_labels)
y_lower = 10

colors = plt.cm.Set2(np.linspace(0, 1, optimal_k))
for i in range(optimal_k):
    cluster_silhouette = sample_silhouette[km_labels == i]
    cluster_silhouette.sort()
    size = len(cluster_silhouette)
    y_upper = y_lower + size

    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_silhouette,
                     facecolor=colors[i], edgecolor=colors[i], alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * size, kmeans_labels.get(i, f"C{i}"), fontsize=9)
    y_lower = y_upper + 10

avg_score = silhouette_score(rfm_scaled, km_labels)
ax.axvline(avg_score, color="#e74c3c", linestyle="--", linewidth=2,
           label=f"Average: {avg_score:.3f}")
ax.set_xlabel("Silhouette Coefficient")
ax.set_ylabel("Cluster")
ax.set_title("Silhouette Plot -- K-Means Clusters", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.set_yticks([])
fig.tight_layout()
save_fig(fig, "16_silhouette_plot.png")
plt.show()


Saved: 16_silhouette_plot.png


## Export Segmented Data

In [17]:
# Save the final segmented customer data
output_cols = ["Customer ID", "recency", "frequency", "monetary",
               "r_score", "f_score", "m_score", "rfm_score",
               "kmeans_cluster", "kmeans_label", "dbscan_cluster"]
rfm_out = rfm[output_cols].copy()

output_path = os.path.join(PROCESSED_DIR, "customer_segments.csv")
rfm_out.to_csv(output_path, index=False)
print(f"Saved segmented customer data: {output_path}")
print(f"Shape: {rfm_out.shape}")
rfm_out.head(10)


Saved segmented customer data: ../data/processed/customer_segments.csv
Shape: (4312, 11)


,Customer ID,recency,frequency,monetary,r_score,f_score,m_score,rfm_score,kmeans_cluster,kmeans_label,dbscan_cluster
0,12346,165,11,372.86,2,5,2,9,0,Dormant,-1
1,12347,3,2,1323.32,5,2,4,11,1,VIP / Champions,0
2,12348,74,1,222.16,2,1,1,4,0,Dormant,1
3,12349,43,3,2671.14,3,3,5,11,1,VIP / Champions,0
4,12351,11,1,300.93,5,1,2,8,0,Dormant,1
5,12352,11,2,343.80,5,2,2,9,0,Dormant,0
6,12353,44,1,317.76,3,1,2,6,0,Dormant,1
7,12355,203,1,488.21,1,1,2,4,0,Dormant,1
8,12356,16,3,3560.30,4,3,5,12,1,VIP / Champions,0
9,12357,24,2,12079.99,4,2,5,11,1,VIP / Champions,-1


## Summary

In [18]:
print("CUSTOMER SEGMENTATION COMPLETE")
print("=" * 55)
print()
print(f"Total customers:  {len(rfm):,}")
print(f"K-Means clusters: {optimal_k}")
print(f"DBSCAN clusters:  {n_clusters} (+ {noise_count} noise points)")
print()
print("K-Means Segments:")
print("-" * 45)
for cid in sorted(kmeans_labels.keys()):
    count = (rfm["kmeans_cluster"] == cid).sum()
    pct = count / len(rfm) * 100
    print(f"  {kmeans_labels[cid]:<25s} {count:>5,} ({pct:.1f}%)")
print()
print(f"Best silhouette score (K-Means): {comparison['K-Means']['silhouette']:.4f}")
print()
print("Next steps:")
print("  - Time-series data preparation for demand forecasting")
print("  - Stationarity tests and seasonal decomposition")


CUSTOMER SEGMENTATION COMPLETE

Total customers:  4,312
K-Means clusters: 2
DBSCAN clusters:  3 (+ 80 noise points)

K-Means Segments:
---------------------------------------------
  Dormant                   2,670 (61.9%)
  VIP / Champions           1,642 (38.1%)

Best silhouette score (K-Means): 0.4223

Next steps:
  - Time-series data preparation for demand forecasting
  - Stationarity tests and seasonal decomposition
